# Quaternion PMD Compensation Results

This notebook compares quaternion-based PMD compensation algorithms (NLMS and Kalman) using the `QuaternionPMDChannel` within the `quantum-comm-sim` framework.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from quantum_comm_sim.channels.quaternion_pmd_channel import QuaternionPMDChannel
from quantum_comm_sim.equalization.quaternion_pmd_equalizer import (
    QuaternionNLMS,
    QuaternionKalmanBaseline,
    TrueQuaternionLMS,
    TrueQuaternionRLS,
    QuaternionMEKF,
    FMQuaternionMEKF,
    compute_broadband_fidelity,
)

results_root = 'results'
DATA_DIR = os.path.join(results_root, 'data')
FIG_DIR = os.path.join(results_root, 'figures')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)


In [ ]:
# Simulation configuration
NUM_FREQ = 5
BANDWIDTH_NM = 40
MAX_ITER = 500
NUM_RUNS = 50
FIBER_LENGTH_KM = 50.0

# Frequency grid (re-using pattern from original notebook)
lambda_0 = 1550e-9
c = 3e8
f_0 = c / lambda_0
delta_lambda = BANDWIDTH_NM * 1e-9 / NUM_FREQ
delta_f = c * delta_lambda / (lambda_0**2)
omega_grid = 2 * np.pi * np.linspace(
    f_0 - delta_f * (NUM_FREQ//2),
    f_0 + delta_f * (NUM_FREQ//2),
    NUM_FREQ
)

# Target quaternion (identity) and spectral weights
q_target = np.array([[1.0], [0.0], [0.0], [0.0]])
spectral_weights = np.ones(NUM_FREQ) / NUM_FREQ


In [ ]:
# Helper: run one Monte Carlo simulation for a given algorithm class
def run_algorithm(AlgClass, label):
    mse_runs = []
    fid_runs = []
    for run in range(NUM_RUNS):
        channel = QuaternionPMDChannel(fiber_length_km=FIBER_LENGTH_KM)
        alg = AlgClass(numfreq=NUM_FREQ)
        mse_hist = []
        fid_hist = []
        for it in range(MAX_ITER):
            channel.update(it)
            state_vec = np.zeros((4, NUM_FREQ))
            for idx, omega in enumerate(omega_grid):
                q = channel.get_state(omega)
                state_vec[:, idx] = q.to_array()
            target_tile = np.tile(q_target, (1, NUM_FREQ))
            alg.adapt(state_vec, target_tile, spectralweights=spectral_weights)
            mse_hist.append(alg.mse_history[-1])
            fid_hist.append(alg.spectral_fidelity_history[-1])
        mse_runs.append(mse_hist)
        fid_runs.append(fid_hist)
    mse_runs = np.array(mse_runs)
    fid_runs = np.array(fid_runs)
    np.save(os.path.join(DATA_DIR, f'mse_{label}.npy'), mse_runs)
    np.save(os.path.join(DATA_DIR, f'fid_{label}.npy'), fid_runs)
    return mse_runs, fid_runs


In [ ]:
# Run selected algorithms: QLMS, QRLS, MEKF, FM-MEKF
labels = ['qlms', 'qrls', 'qmekf', 'fm_qmekf']
classes = [TrueQuaternionLMS, TrueQuaternionRLS, QuaternionMEKF, FMQuaternionMEKF]
all_stats = {}
for label, cls in zip(labels, classes):
    print(f'Running {label} ...')
    if label == 'fm_qmekf':
        alg = cls(numfreq=NUM_FREQ, fast_bins=list(range(NUM_FREQ)), fast_factor=10.0)
        def run_instance():
            return run_algorithm(lambda numfreq=NUM_FREQ: alg, label)
    mse_runs, fid_runs = run_algorithm(cls, label)
    mse_mean = mse_runs.mean(axis=0)
    fid_mean = fid_runs.mean(axis=0)
    all_stats[label] = {'mse_mean': mse_mean, 'fid_mean': fid_mean}


In [ ]:
# Visualization
plt.figure(figsize=(12,5))
for label, color in [('qlms','tab:blue'), ('qrls','tab:orange'), ('qmekf','tab:green'), ('fm_qmekf','tab:red')]:
    plt.semilogy(all_stats[label]['mse_mean'], label=label.upper(), color=color)
plt.xlabel('Iteration')
plt.ylabel('MSE')
plt.grid(True, which='both', alpha=0.3)
plt.legend()
plt.tight_layout()
fig_path = os.path.join(FIG_DIR, 'quaternion_pmd_mse_comparison.png')
plt.savefig(fig_path, dpi=300)
fig_path
